# Karoo TPMS Sensor Analysis
### Capture logs
* enable developer options on Karoo: https://support.hammerhead.io/hc/en-us/articles/30696553134363-Karoo-Enabling-Developer-Options
* get an adb (Android Debugging Bridge) executable: https://www.xda-developers.com/install-adb-windows-macos-linux/#how-to-set-up-adb-on-microsoft-windows
* `adb logcat -b all -v threadtime > karoo.log` from the terminal to capture logs to the file `karoo.log`; stop with [Ctrl]+[C]

### ℹ️ Notebook Info
* Install [DataWrangler](https://marketplace.visualstudio.com/items?itemName=ms-toolsai.datawrangler) extension to view pandas data frames
* install python dependencies with `pip install -r requirements.txt`
* Set ID of the sensor to track below in `ant_id`
* Set path to the logfile below in `logfile_path`

In [4]:
# relative path to logfile
logfile_path = './karoo-tyrewiz-pairing-20260312.log'
# sensor ANT ID
ant_id = 43742


# # relative path to logfile
# logfile_path = './karoo.log'
# # sensor ANT ID
# ant_id = 32155

In [2]:
import pandas as pd
import re
from datetime import datetime

from lark import Lark, Transformer

import ipytest
import pytest
ipytest.autoconfig()

from tqdm import tqdm
tqdm.pandas(ncols=100)
# from tqdm.notebook import tqdm
# # tqdm notebook needs ipywidgets; https://ipywidgets.readthedocs.io/en/stable/user_install.html
# %jupyter nbextension enable --py widgetsnbextension

from tibs import Tibs
from msticpy.vis.timeline import display_timeline

import pprint

In [56]:
def logcat_to_pd(file_path, year = datetime.now().year):
    # ^(\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{3})  -> date and time
    # \s+(\d+)                                  -> pid
    # \s+(\d+)                                  -> tid
    # \s+([VDIWEF])                             -> priority
    # \s+(.*?)                                  -> tag (non-greedy match)
    # :\s(.*)$                                  -> message (everything after colon)
    log_pattern = re.compile(r'^(\d{2}-\d{2}\s\d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\d+)\s+([VDIWEF])\s+(.*?):\s(.*)$')

    parsed_data = []
    discarded_data = []

    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            match = log_pattern.match(line)
            if match:
                parsed_data.append(match.groups())
            else:
                discarded_data.append(line)

    # Create DataFrame
    columns = ['TimeGenerated', 'PID', 'TID', 'Priority', 'Tag', 'Message']
    df = pd.DataFrame(parsed_data, columns=columns)

    # Convert types for better analysis
    df['PID'] = pd.to_numeric(df['PID'])
    df['TID'] = pd.to_numeric(df['TID'])

    df['TimeGenerated'] = pd.to_datetime(f'{year}-' + df['TimeGenerated'], format='%Y-%m-%d %H:%M:%S.%f')
    df['Tag'] = df['Tag'].str.strip()
    # do not use as index, as there are duplicate timestamps
    # df.set_index('TimeGenerated', inplace=True)
    
    return df, discarded_data


# def print_log_df_styled(df):
#     return df.style.set_properties(**{'text-align': 'left'}, subset=['Tag', 'Message', 'Data']) \
#                     .set_properties(**{'text-align': 'right'}, subset=['TimeGenerated', 'PID', 'TID']) \
#                     .set_properties(**{'text-align': 'center'}, subset=['Priority'])

In [4]:
# https://lark-parser.readthedocs.io/en/latest/examples/json_parser.html#sphx-glr-examples-json-parser-py
# https://lark-parser.readthedocs.io/en/latest/json_tutorial.html
# https://github.com/lark-parser/lark/blob/master/examples/advanced/python2.lark
# https://lark-parser.readthedocs.io/en/latest/grammar.html#id1

# grammar summary:
#   input is any string with any number of associative containers embedded
#   valid associative container are:
#       class:      className(key = value, key2 = value)
#       map:        { key = value, key2 = value }
#       jsonlike:   { "key" : value, "key2" : value }
#   start of a container has to be separated from the preceeding text by /\s:;/
#   containers:
#       separator: "," ; assignment: ":" or "="
#       unquoted keys may not contain spaces
#       string values may contain any character except "," unless qouted, or enclosed in matching pair of {[()]}

grammar = r"""
      start                   : root+ | empty_string

      ?root                   : root_string
                              | container_associative

      root_string.0           : root_string_part+
      ?root_string_part       : /.[^\s:;]*/

      ?container.10           : list
                              | container_associative

      ?container_associative.10: map
                              | object
      
      list                    : "[" list_items "]"
      map                     : "{" map_items "}"
      object                  : name "(" map_items ")"

      ?name                   : /[_a-zA-Z]\w*/
      ?key                    : /([_a-zA-Z]\w*)|(\d+)/

      list_items              : [value ("," value)*]
      map_items               : [pair ("," pair)*]
      ?pair                   : key "=" value
                              | ESCAPED_STRING ":" value

      ?value                  : container
                              | scalar
                              | string

      ?scalar                 : SIGNED_NUMBER
                              | "null"        -> null
                              | "true"        -> true
                              | "false"       -> false
                              | empty_string  -> null
      empty_string            :
    
      string                  : outer_part+
      ?outer_part             : ESCAPED_STRING
                              | str_no_special_chars
                              | balanced_parens
                              | balanced_brackets
                              | balanced_braces

      inner_string            : inner_part+
      ?inner_part             : ESCAPED_STRING
                              | str_allow_delimiters
                              | balanced_parens
                              | balanced_brackets
                              | balanced_braces

      balanced_parens         : "(" [inner_string] ")"
      balanced_brackets       : "[" [inner_string] "]"
      balanced_braces         : "{" [inner_string] "}"

      ?str_no_special_chars   : /[^()\[\]{},"]+/
      ?str_allow_delimiters   : /[^()\[\]{}'"]+/

                                                            
      %import common.ESCAPED_STRING
      %import common.SIGNED_NUMBER
      %import common.WS
      %ignore WS
"""

class LogDataTransformer(Transformer):
      def _fix_string(self, str):
            res = str.encode('raw_unicode_escape').decode('unicode_escape')
            if res.startswith(('"', "'")) and res.endswith(('"', "'")):
                  res = res[1:-1]
            return res.strip()

      def _aggregate_string(self, items):
            return self._fix_string("".join(str(i) for i in items if i is not None))

      def start(self, items):
            return [i for i in items if i != ""]

      def root_string(self, items):
            return self._aggregate_string(items)

      def string(self, items):
            return self._aggregate_string(items)

      def inner_string(self, items):
            return self._aggregate_string(items)

      def balanced_parens(self, items): return f"({items[0] or ''})"
      def balanced_brackets(self, items): return f"[{items[0] or ''}]"
      def balanced_braces(self, items): return f"{{{items[0] or ''}}}"

      def SIGNED_NUMBER(self, n):
            return float(n) if '.' in n else int(n)

      def null(self, _): return None
      def true(self, _): return True
      def false(self, _): return False
      def empty_string(self, _): return ""

      def list(self, items):
            return items[0].children if (
                  items[0] is not None and
                  items[0].children is not None and
                  len(items[0].children) != 0 and
                  items[0].children[0] is not None
                  ) else []

      def map(self, items):
            return dict(self.list(items))

      def object(self, items):
            name = str(items[0])
            data = self.map([items[1]])
            data["__name__"] = name
            return data

      def pair(self, items):
            key = self._fix_string(str(items[0]))
            return (key, items[1])
      
      
def parse_log_message(message): # -> []
      return LogDataTransformer().transform(Lark(grammar, parser='earley', ambiguity='resolve').parse(message))


In [5]:
%%ipytest

@pytest.mark.parametrize('input_str, expected', [
    ('className()', [{'__name__': 'className'}]),
    ('"12345-67890)"', ['12345-67890)']),
    ('12345-67890)', ['12345-67890)']),
    ('className(foo=BAR, fizz=buzz, num=12345.67890)', [{'__name__': 'className', 'foo': 'BAR', 'fizz': 'buzz', 'num': 12345.67890}]),
    ('{ foo = BAR , fizz = buzz, num=12345.67890}', [{'foo': 'BAR', 'fizz': 'buzz', 'num': 12345.67890}]),
    ('{"foo": BAR ,"fizz" : buzz, "num": 12345.67890}', [{'foo': 'BAR' ,'fizz': 'buzz', 'num': 12345.67890}]),
    ('{foo: BAR ,fizz : buzz, num=12345.67890}', ['{foo: BAR ,fizz : buzz, num=12345.67890}']),
    ('className(member=className(), other=newClassName(with=value))', [{'__name__': 'className', 'member': {'__name__': 'className'}, 'other': {'__name__': 'newClassName', 'with': 'value'}}]),
    ('12345.67890', ['12345.67890']),
    ('[3.14, true, false, null]', ['[3.14, true, false, null]']),
    ('{list=[3.14, true, false, null]}', [{'list': [3.14, True, False, None]}]),
    ('stringlike, with = all : separators and [] list', ['stringlike, with = all : separators and [] list']),
    ('stringlike className() stringlike', ['stringlike', {'__name__': 'className'}, 'stringlike']),
    ('stringlike : className(foo=BAR, fizz=buzz, num=12345.67890)', ['stringlike :', {'__name__': 'className', 'foo': 'BAR', 'fizz': 'buzz', 'num': 12345.67890}]),
    ('', []),
    # ('', []),
])
def test_parser(input_str, expected):
    assert LogDataTransformer().transform(Lark(grammar, parser='earley', ambiguity='resolve').parse(input_str)) == expected


...............                                                                              [100%]
15 passed in 0.88s


In [6]:
# extract structured data parsed from messages

def parse_antradio(ant_message):
    # full messages in karoo logs have 11 bytes for tx and 19 bytes for rx
    
    # target_ant_tx_re = r'Tx\s*(\[[0-9A-Fa-f]{2}\]){11}\s*$'
    # target_ant_rx_re = r'Rx\s*(\[[0-9A-Fa-f]{2}\]){19}\s*$'
    target_ant_re = r'^(R|T)x\s*(\[[0-9A-Fa-f]{2}\]){11}((\[[0-9A-Fa-f]{2}\]){8})?\s*$'
    
    if not re.match(target_ant_re, ant_message):
        return None

    ant_msg_raw = Tibs.from_hex(''.join(re.findall(r'\[([0-9A-Fa-f]{2})\]', ant_message)))
    
    return {
        'page_no': ant_msg_raw[24:32].u,
        'payload': ant_msg_raw[32:88],
        'raw': ant_msg_raw[24:88],
        'direction': ant_message[0:2].lower()
    }

# def event_antradio(data):
#     return {
#         'page_no': data['page_no'],
#         'direction': data['direction'],
#         'payload_hex': [Tibs.from_u(e, 8).hex for e in list(data['payload'].bytes)],
#         'payload_int': [Tibs.from_u(e, 8).u for e in list(data['payload'].bytes)],
#         'payload_bin': [Tibs.from_u(e, 8).bin for e in list(data['payload'].bytes)]
#     }


def parse_sensorservice(data):
    if type(data[1]) != dict or data[1].get('device') == None:
        return None
    else:
        return data[1]
    
def event_sensorservice(data):
    device_event = {
        'event': data['__name__'],
        'device_id': data['device']['uid']
    }

    if device_event['event'] == 'ChangeDevice':
        device_event = {
            **device_event,
            **{
                'alerts': data['device']['alerts'],
                'tirePressureAlerts': data['device']['tirePressureAlerts'],
                'supportedDataTypes': data['device']['supportedDataTypes']
            }
        }

    return device_event


def parse_hhapp(data):
    if type(data[0]) != dict or data[0].get('eventType') == None:
        return None
    else:
        return data[0]

def event_hhapp(data):
    return { 'event': ' '.join([e for e in [data['eventType'], data['meta'].get('position')] if e != None]) }


def parse_activitytaskmanager(data):
    if type(data[1]) != dict or data[1].get('cmp') == None:
        return None
    else:
        return data[1]

def event_activitytaskmanager(data):
    return { 'event': re.match(r'.*/\.(\S+)', data['cmp']).groups()[0] }
    

## tags filter
* `SensorService`
* `HHApp   `
* `ActivityTaskManager`
* `antradio`


### SensorService
* Tag: `SensorService`; identifies as `d` in some logs
* Level: *all*

#### Events
* Device Lifecycle
  * `AddDevice` (dispatch): single device
  * `ConnectDevice`
  * `ChangeDevice` (dispatch): targets single device
  * `DisableDevice`
  * `RemoveDevice` (dispatch): single device
  * `ForgetDevice`

* `UpdateDevices` (dispatch): target multiple devices
* `SensorServiceModel` (Model updated): overflows log buffer
* `DevicesChanged` (receive): overflows log buffer



### HHApp
*HammerHead App*

* Tag: `r'^HHApp'`
* Level: `VDI`

#### sensor configuration flow
* `Data` has dict with member `source=SENSORS_APP`
* `Message` contains `{"source":"SENSORS_APP","flow":"SENSOR_PAIRING_FLOW",`

#### tire pressure alarms
* `Data` has dict with member `flow=TIRE_PRESSURE`
* `Message` contains `{"source":"ACTIVITY_SERVICE","flow":"TIRE_PRESSURE",`

#### sensor state


### ActivityTaskManager
Logs user interaction with the device.
* Tag: `ActivityTaskManager`
* Level: `I`
* `Data` has dict with member `cmp` starts with `io.hammerhead.sensorsapp/`
* OR `Message` includes `cmp=io.hammerhead.sensorsapp/` pre parsing



### antradio
Logs most raw ANT packets.
* Tag: `antradio`
* Level: *all*; traffic on `W`
* Includes minimum 8 hex "numbers": `r'(\[[0-9A-Fa-f]{2}\]){8,}'`

In [ ]:

ant_id_hex = '{0:04x}'.format(ant_id).upper()

# read logfile into pandas dataframe
raw_df, discard = logcat_to_pd(logfile_path)

# build initial filtered dataframe
df = raw_df[(
        # raw_df['Tag'].str.startswith('SensorService') & 
        raw_df['Message'].str.contains('uid={}'.format(ant_id)) & (
            raw_df['Message'].str.contains('AddDevice') |
            raw_df['Message'].str.contains('ChangeDevice') |
            raw_df['Message'].str.contains('RemoveDevice')
        )
    ) | (
        raw_df['Tag'].str.startswith('HHApp') &
        raw_df['Priority'].str.contains('[VDI]', case=False, regex=True) & (
            raw_df['Message'].str.contains('SENSORS_APP.*SENSOR_', regex=True) |
            raw_df['Message'].str.contains('ACTIVITY_SERVICE.*TIRE_PRESSURE', regex=True)
        )
    ) | (
        raw_df['Tag'].str.startswith('ActivityTaskManager') &
        raw_df['Priority'].str.contains('[I]', case=False, regex=True) &
        raw_df['Message'].str.contains('cmp=io.hammerhead.sensorsapp/')
    ) | (
        raw_df['Tag'].str.startswith('antradio') &
        (
            raw_df['Message'].str.contains(r'Tx\s*(\[[0-9A-Fa-f]{2}\]){11}\s*$', regex=True, case=False) |
            raw_df['Message'].str.contains(r'Rx\s*(\[[0-9A-Fa-f]{{2}}\]){{12}}\[{}\]\[{}\](\[[0-9A-Fa-f]{{2}}\]){{5}}\s*$'.format(ant_id_hex[2:4], ant_id_hex[0:2]), regex=True, case=False)
        )
    )]

# sensor ServiceService seems to log with different tags; rebuild
df.loc[(df['Message'].str.contains('AddDevice') | df['Message'].str.contains('ChangeDevice') | df['Message'].str.contains('RemoveDevice')), 'Tag'] = 'SensorService'

# extract embedded structured data from log messages
df.loc[df['Tag']!='antradio','Data'] = df.loc[df['Tag']!='antradio','Message'].progress_apply(parse_log_message)

df.loc[df['Tag']=='antradio','Data'] = df.loc[df['Tag']=='antradio', 'Message'].apply(parse_antradio)
df.loc[df['Tag']=='SensorService','Data'] = df.loc[df['Tag']=='SensorService','Data'].apply(parse_sensorservice)
df.loc[df['Tag']=='HHApp','Data'] = df.loc[df['Tag']=='HHApp','Data'].apply(parse_hhapp)
df.loc[df['Tag']=='ActivityTaskManager','Data'] = df.loc[df['Tag']=='ActivityTaskManager','Data'].apply(parse_activitytaskmanager)

# cleanup
df = df[df['Data'].notna()]


In [ ]:
## build event dataframe
event_df = df[df['Tag'] != 'antradio']
event_df.loc[event_df['Tag']=='SensorService','Event'] = event_df.loc[event_df['Tag']=='SensorService','Data'].apply(event_sensorservice)
event_df.loc[event_df['Tag']=='HHApp','Event'] = event_df.loc[event_df['Tag']=='HHApp','Data'].apply(event_hhapp)
event_df.loc[event_df['Tag']=='ActivityTaskManager','Event'] = event_df.loc[event_df['Tag']=='ActivityTaskManager','Data'].apply(event_activitytaskmanager)

# printable event string
event_df['EventStr'] = event_df['Event'].apply(str)

# clean df
event_df = event_df.loc[:,['TimeGenerated', 'EventStr', 'Tag', 'Data', 'Event']]

event_df

In [ ]:
## build ant message dataframe
ant_df = df[df['Tag'] == 'antradio']
ant_df['PageNo'] = ant_df['Data'].apply(lambda d: d['page_no'])
ant_df['DIR'] = ant_df['Data'].apply(lambda d: d['direction'])
ant_df['BIN'] = ant_df['Data'].apply(lambda d: [Tibs.from_u(e, 8).bin for e in list(d['payload'].bytes)])
ant_df['HEX'] = ant_df['Data'].apply(lambda d: [Tibs.from_u(e, 8).hex for e in list(d['payload'].bytes)])
ant_df['INT'] = ant_df['Data'].apply(lambda d: [Tibs.from_u(e, 8).u for e in list(d['payload'].bytes)])

# clean df
ant_df = ant_df.loc[:,['TimeGenerated', 'DIR', 'PageNo', 'HEX', 'INT', 'BIN', 'Data']]

ant_df

In [60]:
display_timeline(
   data=ant_df,
   title="Karoo TPMS Comms Logs",
   source_columns=['DIR', 'PageNo', 'HEX', 'INT', 'BIN'],
   group_by='DIR',
   yaxis=False,
   ygrid=False,
   ref_events=event_df,
   ref_col='EventStr',
   legend="left",
   height=200,
   width=1200,
)

Loading BokehJS ...

Column(id='p6407', ...)